# Observation-induced identifiability limits

Known latent OU timescales and directed delays are passed through a candidate first-order observation kernel, 0.10 s integration averaging, and 4.146 Hz sampling. The sampling rate and candidate integration window come from face-video synchronization pulses. This is a sensitivity analysis, not an assertion that the traces are calcium fluorescence.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from yumeow_plot import get_fig

ROOT = Path.cwd()
if ROOT.name == 'analysis' and ROOT.parent.name == 'agent_workspace':
    ROOT = ROOT.parent.parent
INPUT = ROOT / 'agent_workspace/logs/scripts/observation_identifiability/initial'
OUTPUT = ROOT / 'figures/261027观测过程与可辨识性'
OUTPUT.mkdir(parents=True, exist_ok=True)
timescale = pd.read_csv(INPUT / 'timescale_recovery.csv')
delay = pd.read_csv(INPUT / 'delay_recovery.csv')

In [2]:
colors = {0.0: '#000000', 0.2: '#E69F00', 0.5: '#0072B2', 1.0: '#009E73'}
fig, axes = get_fig(RN=1, CN=2, AW=4, AH=4, LM=5, RM=5, TM=5, BM=5, VS=5, HS=5, fontsize=14, dpi=300, lw=1)
ax_loader = iter(np.asarray(axes).reshape(-1))
ax = next(ax_loader)
part = timescale[timescale.exposure_seconds == 0.10]
for tau_calcium, group in part.groupby('tau_calcium_seconds'):
    stats = group.groupby('tau_neural_seconds').apparent_tau_seconds.agg(['median', lambda x: x.quantile(.1), lambda x: x.quantile(.9)]).reset_index()
    ax.plot(stats.tau_neural_seconds, stats['median'], marker='o', color=colors[tau_calcium], label=fr'$\tau_c={tau_calcium:g}$ s')
    ax.fill_between(stats.tau_neural_seconds, stats['<lambda_0>'], stats['<lambda_1>'], color=colors[tau_calcium], alpha=.12)
ax.plot([.1, 4], [.1, 4], ls='--', color='#777777', label='Identity')
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel('True neural timescale (s)')
ax.set_ylabel('Naive apparent timescale (s)')
ax.set_title('4.146 Hz + 0.10 s candidate window')
ax.legend(frameon=False, fontsize=8)
ax = next(ax_loader)
part = delay[delay.sampling_hz == 4]
for tau_calcium, group in part.groupby('tau_calcium_seconds'):
    stats = group.groupby('true_delay_seconds').estimated_delay_seconds.median().reset_index()
    ax.plot(stats.true_delay_seconds, stats.estimated_delay_seconds, marker='o', color=colors[tau_calcium], label=fr'$\tau_c={tau_calcium:g}$ s')
latent = delay[delay.sampling_hz == 100].groupby('true_delay_seconds').estimated_delay_seconds.median().reset_index()
ax.plot(latent.true_delay_seconds, latent.estimated_delay_seconds, marker='.', ls=':', color='#CC79A7', label='100 Hz latent')
ax.plot([0, 1], [0, 1], ls='--', color='#777777')
ax.set_xlabel('Injected delay (s)')
ax.set_ylabel('Peak-lag estimate (s)')
ax.set_title('Delay quantization at 4.146 Hz')
ax.legend(frameon=False, fontsize=8)
for suffix in ('png', 'svg', 'pdf'):
    fig.savefig(OUTPUT / f'observation_identifiability.{suffix}', bbox_inches='tight')
plt.show()

## Interpretation

A naive AR(1) timescale is inflated by integration averaging and an assumed low-pass observation cascade: for example, a true 0.5 s OU process appears as roughly 0.58 s without extra decay and roughly 2.76 s with a 0.5 s decay. Delays below one 4.146 Hz frame collapse to 0 or 0.241 s bins; a fitted one-frame lag therefore cannot distinguish many underlying sub-frame delays. The 0.10 s light-pulse width is an optical proxy, not formal hardware documentation.